In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flammkuchen as fl
from pathlib import Path
from scipy.signal import convolve

In [ ]:
from glob import glob

In [ ]:
def extract_flexible_trial_windows(neural_data, regressors, roi_index, 
                                   pre_seconds=10, 
                                   post_seconds=30, 
                                   sampling_rate=3.0):
    """
    Extract trial windows with flexible boundary handling
    
    Parameters:
    - neural_data: 2D array (ROIs x Time points)
    - regressors: dictionary with 'left_regressor' and 'right_regressor'
    - roi_index: index of ROI to extract
    - pre_seconds: seconds before trial start to extract
    - post_seconds: seconds after trial start to extract
    - sampling_rate: imaging frame rate (Hz)
    
    Returns:
    - Dictionary with left and right trial responses
    """
    # Convert seconds to frames
    pre_frames = int(pre_seconds * sampling_rate)
    post_frames = int(post_seconds * sampling_rate)
    total_frames = pre_frames + post_frames

    # Extract ROI trace
    roi_trace = neural_data[roi_index]
    print(f"\n=== Flexible Trial Extraction for ROI {roi_index} ===")
    print(f"ROI trace length: {len(roi_trace)} frames")
    print(f"Extraction parameters:")
    print(f"  Pre-trial frames: {pre_frames}")
    print(f"  Post-trial frames: {post_frames}")
    print(f"  Total extraction window: {total_frames} frames")

    # Find trial starts for each direction
    def find_trial_starts(regressor):
        """Find trial start frames"""
        trial_starts = np.where((regressor[:-1] == 0) & (regressor[1:] == 1))[0] + 1
        return trial_starts

    def extract_trial_windows(trial_starts):
        """
        Extract trial windows with flexible boundary handling
        
        Modifications:
        - Extract partial trials if full window can't be captured
        - Pad with last available value if needed
        """
        trial_windows = []
        trial_details = []
        
        for i, start in enumerate(trial_starts):
            # Compute window boundaries
            window_start = max(0, start - pre_frames)
            window_end = min(len(roi_trace), start + post_frames)
            
            # Extract available window
            trial_window = roi_trace[window_start:window_end]
            
            # Detailed trial extraction diagnostics
            print(f"\nProcessing Trial {i} (start frame {start}):")
            print(f"  Extraction window: {window_start} to {window_end}")
            print(f"  Available pre-frames: {start - window_start}")
            print(f"  Available post-frames: {window_end - start}")
            
            # If window is shorter than expected, pad with last value
            if len(trial_window) < total_frames:
                padding_needed = total_frames - len(trial_window)
                pad_value = trial_window[-1]  # Use last value for padding
                trial_window = np.pad(trial_window, 
                                      (0, padding_needed), 
                                      mode='constant', 
                                      constant_values=pad_value)
                
                print("  Padded trial to full length")
            
            # Diagnostic information
            print("  Extraction details:")
            print(f"    Final window length: {len(trial_window)}")
            print(f"    Mean value: {np.mean(trial_window)}")
            print(f"    Std value: {np.std(trial_window)}")
            print(f"    Min value: {np.min(trial_window)}")
            print(f"    Max value: {np.max(trial_window)}")
            
            trial_windows.append(trial_window)
            trial_details.append({
                'trial_number': i,
                'start_frame': start,
                'window_start': window_start,
                'window_end': window_end,
                'original_length': len(roi_trace[window_start:window_end]),
                'final_length': len(trial_window)
            })
        
        return np.array(trial_windows), trial_details

    # Extract trials for both directions
    left_trial_starts = find_trial_starts(regressors['left_regressor'])
    right_trial_starts = find_trial_starts(regressors['right_regressor'])

    print("\nTrial Starts:")
    print(f"Left trial starts: {left_trial_starts}")
    print(f"Right trial starts: {right_trial_starts}")

    left_trials, left_details = extract_trial_windows(left_trial_starts)
    right_trials, right_details = extract_trial_windows(right_trial_starts)

    # Final summary
    print("\n=== Extraction Summary ===")
    print(f"Left trials extracted: {len(left_trials)}")
    print(f"Right trials extracted: {len(right_trials)}")

    return {
        'left_trials': left_trials,
        'right_trials': right_trials,
        'left_details': left_details,
        'right_details': right_details
    }

def plot_roi_motion_responses(
    neural_data, 
    regressors, 
    roi_index, 
    output_dir=None, 
    session_name=None, 
    pre_seconds=10, 
    post_seconds=30, 
    sampling_rate=3.0
):
    """
    Generate a two-panel plot showing ROI responses to left and right motion
    
    Parameters:
    - neural_data: 2D array of neural data (ROIs x Time points)
    - regressors: Dictionary with left and right motion regressors
    - roi_index: Index of the ROI to plot
    - output_dir: Directory to save the plot
    - session_name: Optional session identifier
    - pre_seconds: Seconds before trial start to include
    - post_seconds: Seconds after trial start to include
    - sampling_rate: Imaging frame rate (Hz)
    """
    # Print diagnostic information about output directory
    print(f"Attempting to save plot for ROI {roi_index}")
    print(f"Output directory: {output_dir}")
    
    # Ensure output directory exists
    if output_dir:
        import os
        os.makedirs(output_dir, exist_ok=True)
    
    # Convert seconds to frames
    pre_frames = int(pre_seconds * sampling_rate)
    post_frames = int(post_seconds * sampling_rate)
    
    # Find trial starts
    def find_trial_starts(regressor):
        return np.where((regressor[:-1] == 0) & (regressor[1:] == 1))[0] + 1
    
    left_trial_starts = find_trial_starts(regressors['left_regressor'])
    right_trial_starts = find_trial_starts(regressors['right_regressor'])
    
    def extract_trial_responses(trial_starts, roi_trace, pre_seconds=10, post_seconds=30, sampling_rate=3.0):
        """
        Extract trial responses with comprehensive diagnostics
        """
        responses = []
        pre_frames = int(pre_seconds * sampling_rate)
        post_frames = int(post_seconds * sampling_rate)
        total_frames = pre_frames + post_frames

        print("\n=== Trial Response Extraction Diagnostics ===")
        print(f"ROI trace total length: {len(roi_trace)} frames")
        print(f"Extraction parameters:")
        print(f"  Pre-trial frames: {pre_frames}")
        print(f"  Post-trial frames: {post_frames}")
        print(f"  Total frames per trial: {total_frames}")
        print(f"Trial starts: {trial_starts}")

        for i, start in enumerate(trial_starts):
            # Compute window boundaries
            window_start = max(0, start - pre_frames)
            window_end = min(len(roi_trace), start + post_frames)

            print(f"\nProcessing Trial {i}:")
            print(f"  Trial start frame: {start}")
            print(f"  Window start: {window_start}")
            print(f"  Window end: {window_end}")

            # Extract trial window
            trial_response = roi_trace[window_start:window_end]

            # Pad if needed to ensure consistent length
            if len(trial_response) < total_frames:
                padding_needed = total_frames - len(trial_response)
                pad_value = trial_response[-1] if len(trial_response) > 0 else 0
                trial_response = np.pad(trial_response, 
                                        (0, padding_needed), 
                                        mode='constant', 
                                        constant_values=pad_value)

            # Diagnostic information
            print("  Diagnostic information:")
            print(f"    Extracted length: {len(trial_response)}")
            print(f"    Mean value: {np.mean(trial_response)}")
            print(f"    Min value: {np.min(trial_response)}")
            print(f"    Max value: {np.max(trial_response)}")
            print(f"    Contains NaNs: {np.isnan(trial_response).any()}")

            responses.append(trial_response)

        # Convert to numpy array
        responses_array = np.array(responses)
        print("\nFinal responses array:")
        print(f"  Shape: {responses_array.shape}")

        return responses_array
    
    # Extract ROI trace
    roi_trace = neural_data[roi_index]
    
    # Extract responses
    left_responses = extract_trial_responses(left_trial_starts, roi_trace)
    right_responses = extract_trial_responses(right_trial_starts, roi_trace)
    
    # Create time vector
    time_vector = np.linspace(-pre_seconds, post_seconds, pre_frames + post_frames)
    time_vector = np.linspace(-pre_seconds, post_seconds, 120)
    
    print(f"Pre-frames: {pre_frames}")
    print(f"Post-frames: {post_frames}")
    print(f"Time vector length: {len(time_vector)}")
    print(f"Left responses shape: {left_responses.shape}")
    print(f"Right responses shape: {right_responses.shape}")

    # Create figure
    plt.figure(figsize=(12, 6))
    
    # Left motion panel
    plt.subplot(1, 2, 1)
    if len(left_responses) == 10:
        # Average of first 9 trials
        avg_response = np.nanmean(left_responses[:9], axis=0)
        std_response = np.nanstd(left_responses[:9], axis=0) / np.sqrt(9)
        
        plt.fill_between(time_vector, avg_response - std_response, 
                         avg_response + std_response, 
                         color='blue', alpha=0.3)
        plt.plot(time_vector, avg_response, color='blue', alpha=0.7, 
                 label='Average (first 9 trials)')
        
        # Last trial
        plt.plot(time_vector, left_responses[9], color='darkblue', linewidth=2,
                 label='Last trial')
    
    plt.title('Left Motion Responses')
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.legend()
    plt.axvspan(0, 20, alpha=0.2, color='gray')
    plt.axvline(0, color='black', linestyle='--')
    
    # Right motion panel
    plt.subplot(1, 2, 2)
    if len(right_responses) == 10:
        # Average of first 9 trials
        avg_response = np.nanmean(right_responses[:9], axis=0)
        std_response = np.nanstd(right_responses[:9], axis=0) / np.sqrt(9)
        
        plt.fill_between(time_vector, avg_response - std_response, 
                         avg_response + std_response, 
                         color='red', alpha=0.3)
        plt.plot(time_vector, avg_response, color='red', alpha=0.7, 
                 label='Average (first 9 trials)')
        
        # Last trial
        plt.plot(time_vector, right_responses[9], color='darkred', linewidth=2,
                 label='Last trial')
    
    plt.title('Right Motion Responses')
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.legend()
    plt.axvspan(0, 20, alpha=0.2, color='gray')
    plt.axvline(0, color='black', linestyle='--')
    
    plt.tight_layout()
    
    # Save or show plot
    if output_dir:
        # Construct filename
        filename = f"roi_{roi_index}_motion_responses.png"
        full_path = os.path.join(output_dir, filename)
        
        print(f"Saving plot to: {full_path}")
        plt.savefig(full_path)
        print(f"Plot saved successfully to {full_path}")
    else:
        plt.show()
    
    plt.close()




In [ ]:
def find_trial_start_times(regressor, min_gap=30):
    """Find the start times of stimulus trials."""
    # Find transitions from 0 to 1
    transitions = np.where(np.diff(regressor.astype(int)) == 1)[0] + 1
    
    # Filter out transitions that are too close to previous transition
    trial_starts = [transitions[0]]
    for t in transitions[1:]:
        if t - trial_starts[-1] >= min_gap:
            trial_starts.append(t)
    
    return trial_starts

def extract_trial_responses(roi_trace, regressor, pre_frames=30, post_frames=90):
    """Extract trial-by-trial responses for an ROI."""
    # Find trial start times
    trial_starts = find_trial_start_times(regressor)
    
    # Initialize array to store responses
    n_trials = len(trial_starts)
    trial_length = pre_frames + post_frames
    responses = np.zeros((n_trials, trial_length))
    
    # Extract response for each trial
    for i, start in enumerate(trial_starts):
        # Check if we have enough frames before and after
        if start >= pre_frames and start + post_frames <= len(roi_trace):
            trial_slice = slice(start - pre_frames, start + post_frames)
            responses[i, :] = roi_trace[trial_slice]
        else:
            # If trial is at the boundary, fill with NaNs
            responses[i, :] = np.nan
    
    # No longer filter out trials with NaNs - keep all trials
    return responses

def calculate_roi_correlations(neural_data, regressors):
    """
    Calculate correlation between each ROI and motion directions.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary with 'left_regressor' and 'right_regressor'
    
    Returns:
    --------
    dict
        Dictionary with correlation arrays
    """
    n_rois = neural_data.shape[0]
    n_timepoints = neural_data.shape[1]
    
    # Ensure regressors match neural data length
    left_reg = regressors['left_regressor'][:n_timepoints]
    right_reg = regressors['right_regressor'][:n_timepoints]
    
    # Calculate correlations
    left_correlations = np.zeros(n_rois)
    right_correlations = np.zeros(n_rois)
    
    for roi_idx in range(n_rois):
        roi_trace = neural_data[roi_idx]
        left_correlations[roi_idx] = np.corrcoef(roi_trace, left_reg)[0, 1]
        right_correlations[roi_idx] = np.corrcoef(roi_trace, right_reg)[0, 1]
    
    # Handle NaN values
    left_correlations = np.nan_to_num(left_correlations)
    right_correlations = np.nan_to_num(right_correlations)
    
    return {
        'left': left_correlations,
        'right': right_correlations
    }

def plot_roi_motion_responses(neural_data, regressors, roi_index, imaging_rate=3.0, 
                            pre_seconds=10, post_seconds=30, 
                            output_dir=None, session_name="", filename=None):
    """Generate response plots for a specific ROI."""
    print(f"\nAnalyzing ROI {roi_index}")
    

    # Set output paths
    if output_dir is None:
        output_dir = os.getcwd()
    
    if filename is None:
        filename = f"roi_{roi_index}_motion_responses.png"
    
    output_path = os.path.join(output_dir, filename)
    
    # Convert time to frames
    pre_frames = int(pre_seconds * imaging_rate)
    post_frames = int(post_seconds * imaging_rate)
    
    try:
        # Extract regressors
        left_regressor = regressors['left_regressor']
        right_regressor = regressors['right_regressor']
        
        
        # Extract ROI trace from neural data
        if neural_data.shape[0] <= roi_index:
            print(f"  ROI index {roi_index} is out of bounds for neural data with {neural_data.shape[0]} ROIs")
            return None
        
        roi_trace = neural_data[roi_index]
        
        left_starts = find_trial_start_times(left_regressor, min_gap=30)
        right_starts = find_trial_start_times(right_regressor, min_gap=30)
        print(f"  Left trial starts at frames: {left_starts}")
        print(f"  Right trial starts at frames: {right_starts}")
        print(f"  Total recording length: {len(roi_trace)} frames")

        
        # Calculate correlation values for this ROI
        left_corr = np.corrcoef(roi_trace, left_regressor[:len(roi_trace)])[0, 1]
        right_corr = np.corrcoef(roi_trace, right_regressor[:len(roi_trace)])[0, 1]
        left_corr = 0 if np.isnan(left_corr) else left_corr
        right_corr = 0 if np.isnan(right_corr) else right_corr
        
        # Extract trial responses
        left_responses = extract_trial_responses(roi_trace, left_regressor, 
                                               pre_frames=pre_frames, post_frames=post_frames)
        right_responses = extract_trial_responses(roi_trace, right_regressor, 
                                                pre_frames=pre_frames, post_frames=post_frames)
        
        print(f"  Found {len(left_responses)} left trials and {len(right_responses)} right trials")
        
        # After extracting responses:
        print(f"  Left trials with complete data: {len(left_responses)}")
        print(f"  Right trials with complete data: {len(right_responses)}")
        

        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
        
        # Create time vector for x-axis
        time_vector = np.arange(-pre_frames, post_frames) / imaging_rate
        
        # Plot right motion responses (in red)
        if len(right_responses) > 0:
            # For exactly 10 trials (expected case)
            if len(right_responses) == 10:
                # Average of first 9 trials using nanmean to handle NaNs
                avg_response = np.nanmean(right_responses[:9], axis=0)
                std_response = np.nanstd(right_responses[:9], axis=0) / np.sqrt(9)
                
                print(f"  Right motion: using 9 trials for average, plotting trial #9 as last trial")
                print(f"  Right responses shape: {right_responses.shape}")

                axes[0].fill_between(time_vector, avg_response - std_response, 
                                  avg_response + std_response, 
                                  color='red', alpha=0.3)
                axes[0].plot(time_vector, avg_response, color='red', alpha=0.7, 
                           label='Average (first 9 trials)')
                
                # Last trial
                #axes[0].plot(time_vector, right_responses[9], color='darkred', linewidth=2,
                #           label='Last trial')
                axes[0].plot(time_vector, right_responses[9], color='darkred', linewidth=3, 
                           linestyle='--', label='Last trial')
           

            else:
                # For cases with fewer than 10 trials (fallback)
                n_trials = len(right_responses)
                if n_trials > 1:
                    avg_response = np.nanmean(right_responses[:-1], axis=0)
                    std_response = np.nanstd(right_responses[:-1], axis=0) / np.sqrt(n_trials-1)
                    
                    axes[0].fill_between(time_vector, avg_response - std_response, 
                                      avg_response + std_response, 
                                      color='red', alpha=0.3)
                    axes[0].plot(time_vector, avg_response, color='red', alpha=0.7, 
                               label=f'Average (first {n_trials-1} trials)')
                    
                    # Last trial
                    axes[0].plot(time_vector, right_responses[-1], color='darkred', linewidth=2,
                               label='Last trial')
                else:
                    # Single trial case
                    axes[0].plot(time_vector, right_responses[0], color='red', linewidth=2,
                               label='Single trial')
            
            # Add vertical line at stimulus onset
            axes[0].axvline(x=0, color='k', linestyle='--', alpha=0.5)
            
            # Add horizontal line at y=0
            axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
            
            # Shade the stimulus presentation period (20 seconds duration)
            axes[0].axvspan(0, 20, alpha=0.2, color='gray')
            
            axes[0].set_title(f"Rightward Motion (n={len(right_responses)} trials, r={right_corr:.2f})")
        else:
            axes[0].text(0.5, 0.5, "No rightward motion trials found", 
                       ha='center', va='center', transform=axes[0].transAxes)
            axes[0].set_title(f"Rightward Motion (r={right_corr:.2f})")
            axes[0].set_xlim(-pre_seconds, post_seconds)
            axes[0].set_ylim(-0.1, 1.0)
        
        # Plot left motion responses (in blue)
        if len(left_responses) > 0:
            # For exactly 10 trials (expected case)
            if len(left_responses) == 10:
                # Average of first 9 trials using nanmean to handle NaNs
                avg_response = np.nanmean(left_responses[:9], axis=0)
                std_response = np.nanstd(left_responses[:9], axis=0) / np.sqrt(9)
                
                print(f"  Left motion: using 9 trials for average, plotting trial #9 as last trial")
                print(f"  Left responses shape: {left_responses.shape}")
                
                axes[1].fill_between(time_vector, avg_response - std_response, 
                                  avg_response + std_response, 
                                  color='blue', alpha=0.3)
                axes[1].plot(time_vector, avg_response, color='blue', alpha=0.7, 
                           label='Average (first 9 trials)')
                
                # Last trial
                axes[1].plot(time_vector, left_responses[9], color='darkblue', linewidth=3, linestyle='--', label='Last trial')
            
            else:
                # For cases with fewer than 10 trials (fallback)
                n_trials = len(left_responses)
                if n_trials > 1:
                    avg_response = np.nanmean(left_responses[:-1], axis=0)
                    std_response = np.nanstd(left_responses[:-1], axis=0) / np.sqrt(n_trials-1)
                    
                    axes[1].fill_between(time_vector, avg_response - std_response, 
                                      avg_response + std_response, 
                                      color='blue', alpha=0.3)
                    axes[1].plot(time_vector, avg_response, color='blue', alpha=0.7, 
                               label=f'Average (first {n_trials-1} trials)')
                    
                    # Last trial
                    axes[1].plot(time_vector, left_responses[-1], color='darkblue', linewidth=2,
                               label='Last trial')
                else:
                    # Single trial case
                    axes[1].plot(time_vector, left_responses[0], color='blue', linewidth=2,
                               label='Single trial')
            
            # Add vertical line at stimulus onset
            axes[1].axvline(x=0, color='k', linestyle='--', alpha=0.5)
            
            # Add horizontal line at y=0
            axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
            
            # Shade the stimulus presentation period (20 seconds duration)
            axes[1].axvspan(0, 20, alpha=0.2, color='gray')
            
            axes[1].set_title(f"Leftward Motion (n={len(left_responses)} trials, r={left_corr:.2f})")
        else:
            axes[1].text(0.5, 0.5, "No leftward motion trials found", 
                       ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title(f"Leftward Motion (r={left_corr:.2f})")
            axes[1].set_xlim(-pre_seconds, post_seconds)
            axes[1].set_ylim(-0.1, 1.0)
        
        print(f"  Right last trial mean: {np.nanmean(right_responses[9]):.4f}")
        print(f"  Right average mean: {np.nanmean(avg_response):.4f}")
        print(f"  Left last trial mean: {np.nanmean(left_responses[9]):.4f}")
        print(f"  Left average mean: {np.nanmean(avg_response):.4f}")

        # Set axis labels
        for ax in axes:
            ax.set_xlabel("Time from stimulus onset (s)")
            if ax.get_legend_handles_labels()[0]:  # Only add legend if there are items
                ax.legend()
            ax.grid(True, alpha=0.3)
        
        axes[0].set_ylabel("ΔF/F")
        
        # Set title
        direction_preference = ""
        if abs(left_corr - right_corr) >= 0.3:
            if left_corr > right_corr:
                direction_preference = "Left-Selective"
            else:
                direction_preference = "Right-Selective"
        
        if session_name:
            plt.suptitle(f"ROI {roi_index} ({direction_preference}) - Session {session_name}", fontsize=14)
        else:
            plt.suptitle(f"ROI {roi_index} ({direction_preference})", fontsize=14)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=150)
        plt.close()
        
        print(f"  Saved response plot to {os.path.basename(output_path)}")
        
        return output_path
        
    except Exception as e:
        print(f"  Error generating ROI response plot: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def filter_rois_by_correlation(neural_data, regressors, threshold=0.5, direction='both'):
    """
    Filter ROIs based on correlation with motion direction.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary with 'left_regressor' and 'right_regressor'
    threshold : float
        Correlation threshold (0-1)
    direction : str
        'left', 'right', 'both', or 'any'
    
    Returns:
    --------
    list
        Indices of ROIs that meet the criteria
    """
    # Calculate correlations
    correlations = calculate_roi_correlations(neural_data, regressors)
    left_corr = correlations['left']
    right_corr = correlations['right']
    
    # Filter based on direction
    if direction == 'left':
        selected_rois = np.where(left_corr >= threshold)[0].tolist()
    elif direction == 'right':
        selected_rois = np.where(right_corr >= threshold)[0].tolist()
    elif direction == 'both':
        selected_rois = np.where((left_corr >= threshold) & (right_corr >= threshold))[0].tolist()
    elif direction == 'any':
        selected_rois = np.where((left_corr >= threshold) | (right_corr >= threshold))[0].tolist()
    elif direction == 'left_selective':
        selected_rois = np.where((left_corr >= threshold) & (right_corr < threshold))[0].tolist()
    elif direction == 'right_selective':
        selected_rois = np.where((right_corr >= threshold) & (left_corr < threshold))[0].tolist()
    elif direction == 'direction_selective':
        selected_rois = np.where(np.abs(left_corr - right_corr) >= threshold)[0].tolist()
    else:
        raise ValueError(f"Unknown direction: {direction}")
    
    return selected_rois

def plot_correlation_scatter(neural_data, regressors, output_dir=".", session_name="", threshold=0.5):
    """
    Plot scatter of left vs right correlations for all ROIs.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary with 'left_regressor' and 'right_regressor'
    output_dir : str
        Directory to save the plot
    session_name : str
        Name of the session for the plot title
    threshold : float
        Correlation threshold to show on plot
    """
    # Calculate correlations
    correlations = calculate_roi_correlations(neural_data, regressors)
    left_corr = correlations['left']
    right_corr = correlations['right']
    
    # Create figure
    plt.figure(figsize=(10, 8))
    
    # Plot scatter
    plt.scatter(left_corr, right_corr, alpha=0.7, s=40)
    
    # Add diagonal line
    max_val = max(np.max(left_corr), np.max(right_corr))
    min_val = min(np.min(left_corr), np.min(right_corr))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5)
    
    # Add threshold lines
    plt.axhline(y=threshold, color='r', linestyle=':', alpha=0.7)
    plt.axvline(x=threshold, color='b', linestyle=':', alpha=0.7)
    
    # Add labels
    plt.xlabel("Correlation with Leftward Motion")
    plt.ylabel("Correlation with Rightward Motion")
    
    # Add title
    if session_name:
        plt.title(f"Direction Selectivity of ROIs - Session {session_name}")
    else:
        plt.title(f"Direction Selectivity of ROIs")
    
    # Add grid
    plt.grid(True, alpha=0.3)
    
    # Add labels for quadrants
    x_pos = min_val + 0.1 * (max_val - min_val)
    y_pos = max_val - 0.1 * (max_val - min_val)
    plt.text(x_pos, y_pos, "Right-Selective", fontsize=10, ha='left', va='top')
    
    x_pos = max_val - 0.1 * (max_val - min_val)
    y_pos = max_val - 0.1 * (max_val - min_val)
    plt.text(x_pos, y_pos, "Both Directions", fontsize=10, ha='right', va='top')
    
    x_pos = min_val + 0.1 * (max_val - min_val)
    y_pos = min_val + 0.1 * (max_val - min_val)
    plt.text(x_pos, y_pos, "Non-Selective", fontsize=10, ha='left', va='bottom')
    
    x_pos = max_val - 0.1 * (max_val - min_val)
    y_pos = min_val + 0.1 * (max_val - min_val)
    plt.text(x_pos, y_pos, "Left-Selective", fontsize=10, ha='right', va='bottom')
    
    # Save figure
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "correlation_scatter.png")
    plt.savefig(output_path, dpi=150)
    plt.close()
    
    print(f"Saved correlation scatter to {os.path.basename(output_path)}")
    
    return correlations

def plot_selective_rois(neural_data, regressors, output_dir=".", session_name="", 
                       correlation_threshold=0.5, n_rois=5):
    """
    Plot responses for ROIs that are selective to each direction.
    
    Parameters:
    -----------
    neural_data : numpy.ndarray
        Neural activity data (ROIs x time points)
    regressors : dict
        Dictionary with 'left_regressor' and 'right_regressor'
    output_dir : str
        Directory to save output plots
    session_name : str
        Name of the session for plot titles
    correlation_threshold : float
        Threshold for correlation filtering
    n_rois : int
        Number of ROIs to plot per category
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Plot correlation scatter
    correlations = plot_correlation_scatter(neural_data, regressors, output_dir, session_name, correlation_threshold)
    left_corr = correlations['left']
    right_corr = correlations['right']
    
    # Get ROIs for each category
    left_selective = filter_rois_by_correlation(neural_data, regressors, correlation_threshold, 'left_selective')
    right_selective = filter_rois_by_correlation(neural_data, regressors, correlation_threshold, 'right_selective')
    both_responsive = filter_rois_by_correlation(neural_data, regressors, correlation_threshold, 'both')
    
    # Sort by correlation strength
    left_selective = sorted(left_selective, key=lambda idx: left_corr[idx], reverse=True)
    right_selective = sorted(right_selective, key=lambda idx: right_corr[idx], reverse=True)
    both_responsive = sorted(both_responsive, key=lambda idx: left_corr[idx] + right_corr[idx], reverse=True)
    
    # Also get direction selective ROIs (high difference between left and right)
    direction_diff = np.abs(left_corr - right_corr)
    direction_selective = np.where(direction_diff >= 0.3)[0].tolist()
    direction_selective = sorted(direction_selective, key=lambda idx: direction_diff[idx], reverse=True)
    
    # Print summary
    print("\nROI Selectivity Summary:")
    print(f"  Left-selective ROIs: {len(left_selective)}")
    print(f"  Right-selective ROIs: {len(right_selective)}")
    print(f"  Both-responsive ROIs: {len(both_responsive)}")
    print(f"  Direction-selective ROIs: {len(direction_selective)}")
    
    # Create subdirectories
    left_dir = os.path.join(output_dir, "left_selective")
    right_dir = os.path.join(output_dir, "right_selective")
    both_dir = os.path.join(output_dir, "both_responsive")
    direction_dir = os.path.join(output_dir, "direction_selective")
    
    os.makedirs(left_dir, exist_ok=True)
    os.makedirs(right_dir, exist_ok=True)
    os.makedirs(both_dir, exist_ok=True)
    os.makedirs(direction_dir, exist_ok=True)
    
    # Plot ROIs
    # Left-selective
    for roi_idx in left_selective[:n_rois]:
        plot_roi_motion_responses(neural_data, regressors, roi_idx, 
                                 output_dir=left_dir, session_name=session_name)
    
    # Right-selective
    for roi_idx in right_selective[:n_rois]:
        plot_roi_motion_responses(neural_data, regressors, roi_idx, 
                                 output_dir=right_dir, session_name=session_name)
    
    # Both-responsive
    for roi_idx in both_responsive[:n_rois]:
        plot_roi_motion_responses(neural_data, regressors, roi_idx, 
                                 output_dir=both_dir, session_name=session_name)
    
    # Direction-selective
    for roi_idx in direction_selective[:n_rois]:
        plot_roi_motion_responses(neural_data, regressors, roi_idx, 
                                 output_dir=direction_dir, session_name=session_name)
    
    # Save ROI lists
    roi_data = {
        'left_selective': left_selective,
        'right_selective': right_selective,
        'both_responsive': both_responsive,
        'direction_selective': direction_selective,
        'left_correlations': left_corr,
        'right_correlations': right_corr
    }
    
    fl.save(os.path.join(output_dir, "roi_selectivity.h5"), roi_data)
    
    return roi_data

In [ ]:
master = Path(r"Z:\Hagar\main\e0020 imaging")

fish_list = list(master.glob("*_v41*"))
fish = fish_list[2]
print(fish)
num_fish = len(fish_list)

In [ ]:
for plane in range(1, 10):

    session_name = '000' + str(plane)
    print(session_name)

    base_dir = str(fish / 'suite2p' / session_name)
    #neural_data = fl.load(Path(base_dir) / 'data_from_suite2p_cells.h5')['traces']
    neural_data = fl.load(Path(base_dir) / 'filtered_traces.h5')['detr']
    regressors = fl.load(Path(base_dir) /  "motion_regressors.h5")
    neural_data = neural_data.T
    
    # Get ROIs with strong correlation to leftward motion (>0.5)
    left_responsive_rois = filter_rois_by_correlation(
        neural_data=neural_data,
        regressors=regressors,
        threshold=0.5,         # Correlation threshold (0.0-1.0)
        direction='right'       # Look for leftward-responsive ROIs
    )

    # Print the indices to see which ROIs were selected
    print(f"Left-responsive ROIs: {left_responsive_rois}")
    

    for roi in left_responsive_rois:

        # Run flexible extraction
        extraction_results = extract_flexible_trial_windows(
            neural_data, 
            regressors, 
            roi, 
            pre_seconds=10, 
            post_seconds=30
        )

        #plot_trial_extraction_results(
        #    extraction_results, 
        #    output_dir=base_dir
        #)

        plot_roi_motion_responses(
            neural_data=neural_data,
            regressors=regressors,
            roi_index=roi,
            output_dir=base_dir
        )